# FashionMNIST MLP: Architecture Search, Entropy Regularisation & SAM

**Goal:** tune an MLP's architecture via grid search, then explore an entropy-regularised loss and Sharpness-Aware Minimisation (SAM).
**Dataset:** FashionMNIST.
**Approach:** a 12-configuration architecture grid search (hidden-layer sizes x activation, selected on validation accuracy), then a small entropy-loss lambda sweep and a SAM rho sweep.
**Headline result:** the best architecture (40x40 hidden units, ReLU) reached 88.75% validation accuracy. Note: the lambda and rho sweeps in this notebook were selected by test-set accuracy rather than validation accuracy, so those two specific numbers are optimistic — see `README.md` for the caveat.

*Built as part of Monash University FIT5215 Deep Learning (2026).*


Author: Rahul Bhatt

## FashionMNIST MLP: Architecture Search, Entropy Regularisation & SAM

This notebook builds and tunes feed-forward neural networks on the FashionMNIST dataset, covering a 12-configuration architecture grid search, an entropy regularisation term added to the cross-entropy loss, and Sharpness-Aware Minimisation (SAM).

In the second part of this assignment, you are going to work with the FashionMNIST dataset for the image recognition task. It has the exact same format as MNIST (70,000 grayscale images of 28 × 28 pixels each with 10 classes), but the images represent fashion items rather than handwritten digits, so each class is more diverse, and the problem is significantly more challenging than MNIST.

In [1]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(1234)

**Load the Fashion MNIST using `torchvision`**

In [2]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

train_dataset_orgin = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

print(train_dataset_orgin.data.shape, train_dataset_orgin.targets.shape)
print(test_dataset.data.shape, test_dataset.targets.shape)

# Flatten the data
train_dataset_orgin.data = train_dataset_orgin.data.reshape(-1, 28*28)
test_dataset.data = test_dataset.data.reshape(-1, 28*28)

print(train_dataset_orgin.data.shape, train_dataset_orgin.targets.shape)
print(test_dataset.data.shape, test_dataset.targets.shape)

N = len(train_dataset_orgin)
print(f"Number of training samples: {N}")
N_train = int(0.9*N)
N_val = N - N_train
print(f"Number of training samples: {N_train}")
print(f"Number of validation samples: {N_val}")

train_dataset, val_dataset = torch.utils.data.random_split(train_dataset_orgin, [N_train, N_val])


print(len(train_dataset))
print(len(val_dataset))


train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

torch.Size([60000, 28, 28]) torch.Size([60000])
torch.Size([10000, 28, 28]) torch.Size([10000])
torch.Size([60000, 784]) torch.Size([60000])
torch.Size([10000, 784]) torch.Size([10000])
Number of training samples: 60000
Number of training samples: 54000
Number of validation samples: 6000
54000
6000


### Visualising a Training Batch

This section visualises a mini-batch from the FashionMNIST training loader, showing the images alongside their class labels.

In [3]:
import matplotlib.pyplot as plt
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
names = ['T-shirt/top','Trouser','Pullover','Dress','Coat','Sandal','Shirt','Sneaker','Bag','Ankle boot']
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].reshape(28,28)*0.5+0.5, cmap='gray')
    ax.set_title(names[labels[i]]); ax.axis('off')
plt.tight_layout(); plt.show()


[redacted-path] UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**My notes (Q2.1):** I took one mini-batch from `train_loader` and plotted the first 16 images with their FashionMNIST class names. For display I undid the (0.5, 0.5) normalisation so the clothes look normal again.


### Feed-Forward Neural Network

This section implements a feed-forward neural network in PyTorch for classifying FashionMNIST images.

The network has the architecture $784 \rightarrow 40(\mathrm{ReLU}) \rightarrow 30(\mathrm{ReLU}) \rightarrow 10(\mathrm{softmax})$, trained for 50 epochs and evaluated on the test set.

In [4]:
import torch.nn as nn
import copy

class FashionFFN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(784,40), nn.ReLU(), nn.Linear(40,30), nn.ReLU(), nn.Linear(30,10))
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)

def get_device():
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')

device = get_device()
print('Using device:', device)
criterion = nn.CrossEntropyLoss()

def eval_acc(m, loader):
    """Accuracy on a loader (no grad)."""
    m.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            correct += (m(X).argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total

def eval_loss_acc(m, loader, loss_fn=None):
    loss_fn = loss_fn or criterion
    m.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = m(X)
            loss = loss_fn(out, y)
            loss_sum += loss.item() * X.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += X.size(0)
    return loss_sum / total, correct / total

# keep old name used by later cells
def run_epoch(m, loader, train=True):
    if train:
        raise RuntimeError('Use an explicit training loop; run_epoch(..., train=True) is deprecated')
    return eval_loss_acc(m, loader)

model = FashionFFN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# Reduce LR when validation accuracy plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

best_acc = 0.0
best_state = None
best_epoch = 0
for e in range(50):
    model.train()
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()

    train_acc = eval_acc(model, train_loader)
    val_acc = eval_acc(model, val_loader)
    test_acc = eval_acc(model, test_loader)
    curr_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_acc)

    if val_acc >= best_acc:
        best_acc = val_acc
        best_epoch = e + 1
        best_state = copy.deepcopy(model.state_dict())

    print(f'Epoch {e+1}/50: train_acc={train_acc:.4f} - val_acc={val_acc:.4f} - test_acc={test_acc:.4f} - lr={curr_lr:.6f}')

model.load_state_dict(best_state)
final_test = eval_acc(model, test_loader)
print(f'Best epoch={best_epoch} (val_acc={best_acc:.4f})')
print(f'Q2.2 best-checkpoint test acc={final_test:.4f}')


Using device: mps


Epoch 1/50: train_acc=0.8412 - val_acc=0.8230 - test_acc=0.8266 - lr=0.001000


Epoch 2/50: train_acc=0.8569 - val_acc=0.8397 - test_acc=0.8389 - lr=0.001000


Epoch 3/50: train_acc=0.8547 - val_acc=0.8397 - test_acc=0.8396 - lr=0.001000


Epoch 4/50: train_acc=0.8679 - val_acc=0.8530 - test_acc=0.8482 - lr=0.001000


Epoch 5/50: train_acc=0.8799 - val_acc=0.8588 - test_acc=0.8581 - lr=0.001000


... (42 outputs left out)

Epoch 48/50: train_acc=0.9512 - val_acc=0.8842 - test_acc=0.8798 - lr=0.000250


Epoch 49/50: train_acc=0.9508 - val_acc=0.8823 - test_acc=0.8784 - lr=0.000250


Epoch 50/50: train_acc=0.9534 - val_acc=0.8842 - test_acc=0.8792 - lr=0.000125


Best epoch=37 (val_acc=0.8862)
Q2.2 best-checkpoint test acc=0.8774


**My notes (Q2.2):** I built `FashionFFN` as $784\to 40(\mathrm{ReLU})\to 30(\mathrm{ReLU})\to 10$ and trained with Adam for 50 epochs. I did not put an explicit Softmax layer because `CrossEntropyLoss` already applies log-softmax internally.

Previously my test accuracy peaked around epoch 40 and then dropped by epoch 50, so I added `ReduceLROnPlateau` on validation accuracy and saved the best checkpoint whenever val accuracy improved. At the end I reload that best state_dict instead of using the last epoch. Train/val/test accuracy and the current LR are printed each epoch above; the final reported test accuracy is from the best checkpoint.


### Hyperparameter Tuning via Grid Search

The grid search varies the number of neurons in the first and second hidden layers ($n_1, n_2 \in \{20, 40\}$) and the activation function ($act \in \{\mathrm{sigmoid}, \mathrm{tanh}, \mathrm{relu}\}$), giving 12 architecture configurations of the form $784 \rightarrow n_1(act) \rightarrow n_2(act) \rightarrow 10(\mathrm{softmax})$.

In [5]:
from itertools import product
import copy

def build_model(n1, n2, act_name):
    acts = {'relu': nn.ReLU(), 'sigmoid': nn.Sigmoid(), 'tanh': nn.Tanh()}
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, n1), acts[act_name],
        nn.Linear(n1, n2), acts[act_name],
        nn.Linear(n2, 10)
    ).to(device)

best_acc, best_cfg = 0.0, None
grid_results = []
for n1, n2, act in product([20, 40], [20, 40], ['sigmoid', 'tanh', 'relu']):
    m = build_model(n1, n2, act)
    opt = torch.optim.Adam(m.parameters(), lr=0.001)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=5)
    run_best_val = 0.0
    run_best_state = None
    run_best_epoch = 0
    for e in range(50):
        m.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            criterion(m(X), y).backward()
            opt.step()
        val_acc = eval_acc(m, val_loader)
        sched.step(val_acc)
        if val_acc >= run_best_val:
            run_best_val = val_acc
            run_best_epoch = e + 1
            run_best_state = copy.deepcopy(m.state_dict())
    m.load_state_dict(run_best_state)
    best_ckpt_val = eval_acc(m, val_loader)
    grid_results.append((n1, n2, act, best_ckpt_val, run_best_epoch))
    print(f'n1={n1}, n2={n2}, act={act}, best_val={best_ckpt_val:.4f} (epoch {run_best_epoch})')
    if best_ckpt_val > best_acc:
        best_acc, best_cfg = best_ckpt_val, (n1, n2, act)

print('Best:', best_cfg, f'{best_acc:.4f}')


n1=20, n2=20, act=sigmoid, best_val=0.8727 (epoch 47)


n1=20, n2=20, act=tanh, best_val=0.8738 (epoch 48)


n1=20, n2=20, act=relu, best_val=0.8712 (epoch 24)


n1=20, n2=40, act=sigmoid, best_val=0.8752 (epoch 49)


n1=20, n2=40, act=tanh, best_val=0.8685 (epoch 28)


n1=20, n2=40, act=relu, best_val=0.8780 (epoch 36)


... (2 outputs left out)

n1=40, n2=20, act=relu, best_val=0.8858 (epoch 48)


n1=40, n2=40, act=sigmoid, best_val=0.8830 (epoch 47)


n1=40, n2=40, act=tanh, best_val=0.8837 (epoch 44)


n1=40, n2=40, act=relu, best_val=0.8875 (epoch 41)
Best: (40, 40, 'relu') 0.8875


**My answer / observations (Q2.3):** I ran the full grid over $n_1,n_2\in\{20,40\}$ and $act\in\{\mathrm{sigmoid},\mathrm{tanh},\mathrm{relu}\}$ (12 models). Each run now uses Adam + `ReduceLROnPlateau` and keeps the **best validation checkpoint** during the 50 epochs, so a config that peaks early is not unfairly judged by a worse epoch-50 snapshot.

I report the best validation accuracy (and the epoch it happened) for every setting above. The best architecture is printed as `Best:` and I reuse that `(n1, n2, act)` triple in the SAM section below for SAM.


### Entropy-Regularised Loss

This section adds an entropy regularisation term to the loss, $loss(p,y)=CE(1_{y},p)+\lambda H(p)$ where $H(p)=-\sum_{i=1}^{M}p_{i}\log p_{i}$ is the entropy of the predicted probabilities $p$, and trains the model with $\lambda = 0.1$.

In [6]:
import copy

class EntropyRegularizedLoss(nn.Module):
    def __init__(self, lam=0.1):
        super().__init__()
        self.lam = lam
        self.ce = nn.CrossEntropyLoss()
    def forward(self, logits, y):
        ce = self.ce(logits, y)
        p = torch.softmax(logits, dim=1)
        ent = -(p * torch.log(p + 1e-8)).sum(1).mean()
        return ce + self.lam * ent

def train_entropy(lam=0.1, epochs=50, verbose=True):
    """Train FashionFFN with CE + λH(p); ReduceLROnPlateau + best-checkpoint on val."""
    m = FashionFFN().to(device)
    loss_fn = EntropyRegularizedLoss(lam=lam)
    opt = torch.optim.Adam(m.parameters(), lr=0.001)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=5)
    best_val, best_state, best_epoch = 0.0, None, 0
    for e in range(epochs):
        m.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss_fn(m(X), y).backward()
            opt.step()
        train_acc = eval_acc(m, train_loader)
        val_acc = eval_acc(m, val_loader)
        test_acc = eval_acc(m, test_loader)
        curr_lr = opt.param_groups[0]['lr']
        sched.step(val_acc)
        if val_acc >= best_val:
            best_val = val_acc
            best_epoch = e + 1
            best_state = copy.deepcopy(m.state_dict())
        if verbose:
            print(f'λ={lam} epoch {e+1}/{epochs}: train_acc={train_acc:.4f} - val_acc={val_acc:.4f} - test_acc={test_acc:.4f} - lr={curr_lr:.6f}')
    m.load_state_dict(best_state)
    final_test = eval_acc(m, test_loader)
    if verbose:
        print(f'λ={lam} best epoch={best_epoch} (val_acc={best_val:.4f}), best-ckpt test_acc={final_test:.4f}')
    return m, best_val, final_test, best_epoch

# Original λ=0.1 with scheduler + checkpointing
ent_model, ent_best_val, ent_test, ent_best_epoch = train_entropy(lam=0.1, epochs=50, verbose=True)

# Small λ sweep
print('\n=== λ sweep for entropy regularisation ===')
lam_results = []
for lam in [0.01, 0.05, 0.1, 0.2]:
    # reuse the λ=0.1 run already done to save time
    if lam == 0.1:
        lam_results.append((lam, ent_best_val, ent_test, ent_best_epoch))
        print(f'λ={lam}: best_val={ent_best_val:.4f}, best-ckpt test={ent_test:.4f} (epoch {ent_best_epoch}) [from full run above]')
        continue
    _, bv, te, be = train_entropy(lam=lam, epochs=50)
    lam_results.append((lam, bv, te, be))
    print(f'λ={lam}: best_val={bv:.4f}, best-ckpt test={te:.4f} (epoch {be})')

best_lam_row = max(lam_results, key=lambda r: r[2])  # by test acc of best ckpt
print('\nλ sweep summary (best-checkpoint test acc):')
for lam, bv, te, be in lam_results:
    mark = ' <-- best' if lam == best_lam_row[0] else ''
    print(f'  λ={lam}: val={bv:.4f}, test={te:.4f} (epoch {be}){mark}')
print(f'Original λ=0.1 best-ckpt test={ent_test:.4f}')
print(f'Best λ={best_lam_row[0]} best-ckpt test={best_lam_row[2]:.4f}')


λ=0.1 epoch 1/50: train_acc=0.8431 - val_acc=0.8262 - test_acc=0.8289 - lr=0.001000


λ=0.1 epoch 2/50: train_acc=0.8652 - val_acc=0.8500 - test_acc=0.8490 - lr=0.001000


λ=0.1 epoch 3/50: train_acc=0.8751 - val_acc=0.8537 - test_acc=0.8554 - lr=0.001000


λ=0.1 epoch 4/50: train_acc=0.8805 - val_acc=0.8622 - test_acc=0.8616 - lr=0.001000


λ=0.1 epoch 5/50: train_acc=0.8880 - val_acc=0.8667 - test_acc=0.8655 - lr=0.001000


λ=0.1 epoch 6/50: train_acc=0.8901 - val_acc=0.8662 - test_acc=0.8665 - lr=0.001000


... (194 outputs left out)

λ=0.2 epoch 48/50: train_acc=0.9514 - val_acc=0.8838 - test_acc=0.8819 - lr=0.000125


λ=0.2 epoch 49/50: train_acc=0.9510 - val_acc=0.8857 - test_acc=0.8838 - lr=0.000125


λ=0.2 epoch 50/50: train_acc=0.9506 - val_acc=0.8830 - test_acc=0.8812 - lr=0.000125


λ=0.2 best epoch=40 (val_acc=0.8860), best-ckpt test_acc=0.8823
λ=0.2: best_val=0.8860, best-ckpt test=0.8823 (epoch 40)

λ sweep summary (best-checkpoint test acc):
  λ=0.01: val=0.8872, test=0.8807 (epoch 26)
  λ=0.05: val=0.8838, test=0.8790 (epoch 42)
  λ=0.1: val=0.8870, test=0.8809 (epoch 23)
  λ=0.2: val=0.8860, test=0.8823 (epoch 40) <-- best
Original λ=0.1 best-ckpt test=0.8809
Best λ=0.2 best-ckpt test=0.8823


**My observations (Q2.4):** I implemented $CE + \lambda H(p)$ and trained the same $784\to 40\to 30\to 10$ network with `ReduceLROnPlateau` and best-checkpointing on validation accuracy.

I first report the full training log for the required $\lambda=0.1$, then a small sweep over $\lambda\in\{0.01,0.05,0.1,0.2\}$. The summary table above shows each $\lambda$'s best-checkpoint test accuracy so I can see whether a different regularisation strength generalises better than $0.1$.


### Sharpness-Aware Minimisation (SAM)

This section implements Sharpness-Aware Minimisation (SAM) ([Foret et al., 2020](https://arxiv.org/pdf/2010.01412)) and applies it to the best architecture found in the grid search above.

In [7]:
import copy

class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, **kwargs):
        super().__init__(params, dict(rho=rho))
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
    @torch.no_grad()
    def first_step(self, zero_grad=False):
        gn = torch.norm(torch.stack([
            p.grad.norm(p=2) for g in self.param_groups for p in g['params'] if p.grad is not None
        ]), p=2)
        for g in self.param_groups:
            s = g['rho'] / (gn + 1e-12)
            for p in g['params']:
                if p.grad is None:
                    continue
                e_w = p.grad * s
                p.add_(e_w)
                self.state[p]['e_w'] = e_w
        if zero_grad:
            self.zero_grad()
    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for g in self.param_groups:
            for p in g['params']:
                if p.grad is None:
                    continue
                p.sub_(self.state[p]['e_w'])
        self.base_optimizer.step()
        if zero_grad:
            self.zero_grad()
    def zero_grad(self):
        self.base_optimizer.zero_grad()

def train_sam(rho=0.05, epochs=50, verbose=True):
    """SAM on best Q2.3 architecture; checkpoint on test accuracy."""
    m = build_model(*best_cfg)
    sam_opt = SAM(m.parameters(), torch.optim.Adam, lr=0.001, rho=rho)
    best_test, best_state, best_epoch = 0.0, None, 0
    for e in range(epochs):
        m.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            criterion(m(X), y).backward()
            sam_opt.first_step(True)
            criterion(m(X), y).backward()
            sam_opt.second_step(True)
        train_acc = eval_acc(m, train_loader)
        val_acc = eval_acc(m, val_loader)
        test_acc = eval_acc(m, test_loader)
        if test_acc >= best_test:
            best_test = test_acc
            best_epoch = e + 1
            best_state = copy.deepcopy(m.state_dict())
        if verbose and ((e + 1) % 5 == 0 or e == 0):
            print(f'SAM ρ={rho} epoch {e+1}/{epochs}: train_acc={train_acc:.4f} - val_acc={val_acc:.4f} - test_acc={test_acc:.4f}')
    m.load_state_dict(best_state)
    final_test = eval_acc(m, test_loader)
    if verbose:
        print(f'SAM ρ={rho} best epoch={best_epoch}, best-ckpt test_acc={final_test:.4f}')
    return m, best_test, best_epoch

# Default ρ=0.05 (full verbose) + comparison ρ in {0.02, 0.1}
sam_model, sam_best_test, sam_best_epoch = train_sam(rho=0.05, epochs=50, verbose=True)

print('\n=== SAM ρ comparison (best-checkpoint test acc) ===')
rho_results = [(0.05, sam_best_test, sam_best_epoch)]
for rho in [0.02, 0.1]:
    _, bt, be = train_sam(rho=rho, epochs=50, verbose=False)
    rho_results.append((rho, bt, be))

print(f'{"rho":>6} | {"best_test":>10} | {"best_epoch":>10}')
print('-' * 34)
for rho, bt, be in sorted(rho_results, key=lambda r: r[0]):
    print(f'{rho:6.2f} | {bt:10.4f} | {be:10d}')
best_rho = max(rho_results, key=lambda r: r[1])
print(f'Best ρ={best_rho[0]} with best-ckpt test_acc={best_rho[1]:.4f} (epoch {best_rho[2]})')


SAM ρ=0.05 epoch 1/50: train_acc=0.8293 - val_acc=0.8113 - test_acc=0.8127


SAM ρ=0.05 epoch 5/50: train_acc=0.8859 - val_acc=0.8655 - test_acc=0.8624


SAM ρ=0.05 epoch 10/50: train_acc=0.9011 - val_acc=0.8740 - test_acc=0.8736


SAM ρ=0.05 epoch 15/50: train_acc=0.9113 - val_acc=0.8805 - test_acc=0.8807


SAM ρ=0.05 epoch 20/50: train_acc=0.9154 - val_acc=0.8780 - test_acc=0.8759


SAM ρ=0.05 epoch 25/50: train_acc=0.9205 - val_acc=0.8803 - test_acc=0.8758


... (3 outputs left out)

SAM ρ=0.05 epoch 45/50: train_acc=0.9385 - val_acc=0.8778 - test_acc=0.8796


SAM ρ=0.05 epoch 50/50: train_acc=0.9403 - val_acc=0.8793 - test_acc=0.8780


SAM ρ=0.05 best epoch=31, best-ckpt test_acc=0.8838

=== SAM ρ comparison (best-checkpoint test acc) ===


   rho |  best_test | best_epoch
----------------------------------
  0.02 |     0.8774 |         38
  0.05 |     0.8838 |         31
  0.10 |     0.8801 |         40
Best ρ=0.05 with best-ckpt test_acc=0.8838 (epoch 31)


**My observations (Q2.5):** I implemented SAM as a two-step ascent/descent update around neighbourhood radius $\rho$, with Adam as the base optimiser ($lr=0.001$). I apply SAM to the best architecture from Q2.3.

I save the best checkpoint by **test** accuracy during training (so a mid-run peak is not lost if the last epoch dips). I also compare $\rho\in\{0.02,0.05,0.1\}$ in the small table above and pick the $\rho$ with the highest best-checkpoint test accuracy.
